In [42]:
# Install Required Libraries
!pip install einops albumentations torchinfo

In [43]:
# Import Libraries
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")


In [44]:
#Import Libraries
import torch
import matplotlib.pyplot as plt

torch.manual_seed(42)

In [45]:
#Create Sample Data
batch_size = 8
features = 6

X = torch.randn(batch_size, features)

print("Input Shape:", X.shape)

Input Shape: torch.Size([8, 6])


In [46]:
#Initialize Learnable Parameters
gamma = torch.ones(features)

beta = torch.zeros(features)

In [47]:
#Batch Normalization (Forward Pass)
def batch_norm_forward(X, gamma, beta, eps=1e-5):

    mean = torch.mean(X, dim=0)

    variance = torch.var(X, dim=0, unbiased=False)

    X_hat = (X - mean) / torch.sqrt(variance + eps)

    output = gamma * X_hat + beta

    cache = (X, X_hat, mean, variance, gamma, beta, eps)


In [48]:
#Layer Normalization (Forward Pass)
def layer_norm_forward(X, gamma, beta, eps=1e-5):

    mean = torch.mean(X, dim=1, keepdim=True)

    variance = torch.var(X, dim=1, keepdim=True, unbiased=False)

    X_hat = (X - mean) / torch.sqrt(variance + eps)

    output = gamma * X_hat + beta

    cache = (X, X_hat, mean, variance, gamma, beta, eps)

    return output, cache

In [49]:
#Run LayerNorm
ln_output, ln_cache = layer_norm_forward(X, gamma, beta)

print(ln_output.shape)

torch.Size([8, 6])


In [50]:
#BatchNorm Backward Pass
def batch_norm_backward(dout, cache):

    X, X_hat, mean, var, gamma, beta, eps = cache

    N = X.shape[0]

    dbeta = torch.sum(dout, dim=0)

    dgamma = torch.sum(dout * X_hat, dim=0)

    dxhat = dout * gamma

    dvar = torch.sum(
        dxhat * (X - mean) * (-0.5) * (var + eps) ** (-1.5),
        dim=0
    )

    dmean = (
        torch.sum(dxhat * (-1 / torch.sqrt(var + eps)), dim=0)
        + dvar * torch.mean(-2 * (X - mean), dim=0)
    )

    dx = (
        dxhat / torch.sqrt(var + eps)
        + dvar * 2 * (X - mean) / N
        + dmean / N
    )

    return dx, dgamma, dbeta

In [51]:
#LayerNorm Backward Pass
def layer_norm_backward(dout, cache):

    X, X_hat, mean, var, gamma, beta, eps = cache

    D = X.shape[1]

    dbeta = torch.sum(dout, dim=0)

    dgamma = torch.sum(dout * X_hat, dim=0)

    dxhat = dout * gamma

    dvar = torch.sum(
        dxhat * (X - mean) * (-0.5) * (var + eps) ** (-1.5),
        dim=1,
        keepdim=True
    )

    dmean = (
        torch.sum(
            dxhat * (-1 / torch.sqrt(var + eps)),
            dim=1,
            keepdim=True
        )
        + dvar * torch.mean(
            -2 * (X - mean),
            dim=1,
            keepdim=True
        )
    )

    dx = (
        dxhat / torch.sqrt(var + eps)
        + dvar * 2 * (X - mean) / D
        + dmean / D
    )

    return dx, dgamma, dbeta